# ASL-Tutor: Webcam Demo & Bandit Evaluation

This notebook demonstrates:
1. Real-time webcam inference
2. Contextual bandit policy evaluation
3. A/B testing simulation

In [ ]:
import os
import sys
import numpy as np
import matplotlib.pyplot as plt

# Add parent directory to path
sys.path.insert(0, os.path.dirname(os.getcwd()))

## 1. Webcam Demo (Optional)

Run this cell to start the webcam demo. Press 'q' to quit.

In [ ]:
# Uncomment to run webcam demo
# from src.inference import run_webcam_demo
# run_webcam_demo(model_path='models/asl_cnn_best.pt')

## 2. Student Model Demo

In [ ]:
from src.student_model import StudentModel
from src.dataset import ASL_CLASSES
import random

# Create a test student
student = StudentModel(user_id='demo_student', data_dir='data/demo_users')
print(f"Created student: {student}")

In [ ]:
# Simulate some learning
print("Simulating 50 practice attempts...\n")

for i in range(50):
    # Pick a sign (biased toward first 10 signs)
    if random.random() > 0.3:
        sign = random.choice(ASL_CLASSES[:10])
    else:
        sign = random.choice(ASL_CLASSES)
    
    # Simulate outcome (easier signs have higher success rate)
    base_prob = 0.5 + 0.04 * student.get_progress(sign).attempts
    correct = random.random() < min(base_prob, 0.95)
    response_time = random.uniform(1.5, 5.0)
    
    student.update(sign, correct, response_time)

print(f"After simulation: {student}")

In [ ]:
# Visualize mastery
mastery = student.get_mastery_summary()

# Sort by mastery
sorted_mastery = sorted(mastery.items(), key=lambda x: x[1], reverse=True)

plt.figure(figsize=(14, 6))
signs = [s for s, _ in sorted_mastery]
values = [v for _, v in sorted_mastery]

colors = ['green' if v >= 0.8 else 'orange' if v >= 0.3 else 'red' for v in values]
plt.bar(signs, values, color=colors)
plt.axhline(y=0.8, color='green', linestyle='--', label='Mastery threshold')
plt.xlabel('Sign')
plt.ylabel('Mastery Level')
plt.title('Student Mastery by Sign')
plt.xticks(rotation=45)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Context features example
print("Context features for sign 'A':")
context = student.get_context('A')
feature_names = ['mastery', 'attempts_norm', 'avg_time_norm', 'days_since_last', 'avg_mastery', 'streak_norm']

for name, value in zip(feature_names, context):
    print(f"  {name}: {value:.4f}")

## 3. Contextual Bandit Evaluation

In [ ]:
from src.bandit import (
    LinearThompsonSampling, 
    LinUCB, 
    RandomPolicy, 
    FixedCurriculumPolicy,
    run_simulation,
    compare_policies
)

In [ ]:
# Test Thompson Sampling
policy = LinearThompsonSampling()
test_student = StudentModel(user_id='policy_test', data_dir='/tmp/policy_test')

print("Thompson Sampling - Sign Selection Demo:")
for i in range(10):
    sign = policy.select_next_sign(test_student)
    print(f"  Step {i+1}: Selected '{sign}'")
    
    # Simulate outcome
    correct = random.random() > 0.4
    time = random.uniform(1, 4)
    context = test_student.get_context(sign)
    reward = 1.0 if correct and time < 3 else 0.0
    
    test_student.update(sign, correct, time)
    policy.update(sign, context, reward)

In [ ]:
# Compare policies with simulation
print("Running policy comparison simulation...")
print("(This may take a minute)\n")

# Run simulations
avg_results = compare_policies(n_steps=300, n_runs=3)

## 4. A/B Testing Simulation

In [ ]:
from src.evaluation import ABTestManager

# Create A/B test manager
ab_manager = ABTestManager(data_dir='data/ab_test_notebook')

print("Running A/B Experiment...")
print("Comparing Fixed Curriculum vs Adaptive Bandit\n")

In [ ]:
# Run experiment
results = ab_manager.run_ab_experiment(
    n_users_per_group=5,
    n_steps_per_user=150,
    student_lr=0.12
)

In [ ]:
# Analyze and plot results
df = ab_manager.analyze_results(results)
ab_manager.plot_results(results)

## 5. Learner Report

In [ ]:
from src.evaluation import print_learner_report, generate_learner_summary

# Print report for our demo student
print_learner_report('demo_student', data_dir='data/demo_users')

## Summary

This notebook demonstrated:

1. **Student Model**: Tracks per-sign mastery with exponential moving averages
2. **Contextual Bandit**: Adaptively selects signs based on student context
3. **Policy Comparison**: Thompson Sampling outperforms random and fixed curricula
4. **A/B Testing**: Framework for comparing learning strategies

Key findings:
- Adaptive policies lead to faster mastery
- The bandit learns to focus on challenging signs
- Spaced repetition effects improve retention